In [ ]:
import warnings
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
# warnings.filterwarnings('ignore')

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_postgres.vectorstores import PGVector


CONNECTION = os.environ.get("PGVT_CONNECTION")

COLLECTION_NAME = "documents"
DOCUMENT_RELEVANT_THRESHOLD = 0.7


llm = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    temperature=0.0,
)

chat_model = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    temperature=0.5,
)

ranking = NVIDIARerank(
    model="nvidia/nv-rerankqa-mistral-4b-v3",
    truncate="END"
)

embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-mistral-7b-v2",
    truncate="END"
)

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION,
    use_jsonb=True,
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class RouteQuery(BaseModel):
    """
    A model representing a routing decision for user queries. 
    Determines whether a query is processed via RAG (Retrieval-Augmented Generation) 
    or handled as a regular chatbot conversation.
    """

    route: Literal["RAG", "Chatbot"] = Field(..., description="The determined route for processing: 'RAG' or 'Chatbot'.")

structured_llm_router = llm.with_structured_output(RouteQuery)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

class RouteQuery(BaseModel):
    """
    A model representing a routing decision for user queries. 
    Determines whether a query is processed via RAG (Retrieval-Augmented Generation) 
    or handled as a regular chatbot conversation.
    """

    route: Literal["RAG", "Chatbot"] = Field(
        ..., description="The determined route for processing: 'RAG' or 'Chatbot'.")


structured_llm_router = llm.with_structured_output(RouteQuery)
system = """
You are a smart router determining whether to process a user's input using the RAG retrieval system or 
handle it as a conversational response from the chatbot. Follow these steps:

  1. Use the RAG process if the input. 
     * Requires detailed factual retrieval from a specific knowledge base or document.
     * Mentions topics not covered by the chatbot's general knowledge or external tools

     Examples include:
      * 'What is the financial projection for Q3 2024?'
      * 'Summarize the company's annual report.'
      * 'Retrieve details about document X.'

  2. Handle it as a chatbot interaction if the input. 
     * Seeks up-to-date information such as weather, current events, or trending topics (use the web search tool as needed).
     * Relates to user-specific references that can be resolved using long-term memory.
     * Involves casual conversation, creative tasks, or opinion-based queries

     Examples include:
      * 'What's the weather like in DaNang today?'
      * 'Tell me a joke.'
      * 'Hi'

Output:
  * If RAG is needed, respond with: "RAG"
  * If it's a regular conversation, respond directly as a "Chatbot"
"""

route_prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{question}")])

route_chain = route_prompt | structured_llm_router

In [ ]:
print(route_chain.invoke({"question": "Base on my financial report of November 2024. Can you tell how should I invest in what Stock"}))

In [ ]:
print(route_chain.invoke({"question": "What my name?"}))

In [ ]:
### Generate

from langchain_core.output_parsers import StrOutputParser

rag_prompt = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n
          Question: {question}\n
          Context: {context}\n
          Answer: """
generate_prompt = ChatPromptTemplate.from_template(rag_prompt)
rag_chain = generate_prompt | chat_model | StrOutputParser()

In [ ]:
### Question Re-writer

from langchain_core.prompts import PromptTemplate

system_prompt = """You a question re-writer that converts an input question to a better version that is optimized,
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning.
     Here is the information you need to know:
     INIT QUESTION: {question}./
     IMPROVED QUESTION: \n

     CAVEAT: Only give back the final question
     """
re_write_prompt = PromptTemplate.from_template(system_prompt)
question_rewriter = re_write_prompt | llm | StrOutputParser()

In [ ]:
from typing import List
from langgraph.graph import MessagesState
from typing_extensions import TypedDict
from langchain.schema import Document


class State(MessagesState):
    pass
    # question: str
    # generation: str
    documents: List[Document]

In [ ]:
### Search

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(k=3)

In [ ]:
# Utils def

# Util def to get all documents content
def format_docs(docs: List[Document]):
    return "\n\n".join(doc.page_content for doc in docs)

# Get user_id from config
from langchain_core.runnables import RunnableConfig
def get_user_id(config: RunnableConfig) -> str:
    user_id = config["configurable"].get("user_id", "")
    if user_id is None:
        raise ValueError("User ID needs to be provided to save a memory.")

    return user_id

# Sigmoid activation function
import numpy as np

def sigmoid(x: float):
    return 1 / (1 + np.exp(-x))

# Example usage:
logits = np.array([0.5, -1.2, 3.0])
probabilities = sigmoid(logits)
print(probabilities)

In [ ]:
### Define Graph Nodes
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool

class RetrieverInput(BaseModel):
    """Input to the retriever."""

    query: str = Field(..., description="A string containing information to look up in retriever")

@tool(args_schema=RetrieverInput)
def retrieve(query: str, config: RunnableConfig) -> List[Document]:
    """
    Use this tool to retrieve uploaded documents. These documents 
    may include financial statistics, user expense reports over specific time periods, 
    or other documents related their personal information.
    """
    user_id = get_user_id(config)
    filter = {"user_id": {"$eq": user_id}}
    search_kwargs = {
        "k": 3,
        "fetch_k": 5,
        # "filter": filter
    }
    retriever = vector_store.as_retriever(search_type="mmr",
                                          search_kwargs=search_kwargs)
    documents = retriever.invoke(query)

    return documents


tools = [retrieve, web_search_tool]

def agent(state: State) -> State:
    """
    Invokes the agent model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply end.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with the agent response appended to messages
    """
    messages = state["messages"]
    
    model = chat_model.bind_tools(tools)
    response = model.invoke(messages)

    # return {"messages": [response], "question": state["messages"][-1]}
    return {"messages": [response]}


def rewrite(state: State) -> State:
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """
    messages = state["messages"]
    question = messages[0].content
    
    improved_question = question_rewriter.invoke({"question": question})

    return {"messages": [improved_question]}


def generate(state: State) -> State: 
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    messages = state["messages"]
    question = messages[0].content
    last_message = messages[-1]

    docs = last_message.content


    generation = rag_chain.invoke({"context": format_docs(docs), "question":  question})

    return {"messages": [generation]}


In [ ]:
### Edges

def grade_documents(state: State) -> Literal["generate", "rewrite"]:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        str: A decision for whether the documents are relevant or not
    """
    messages = state["messages"]
    last_message = messages[-1]

    # question = state["question"]
    # docs = last_message.content
    question = messages[0].content
    docs = last_message.content
    print(docs)

    # Using ranking model to filter irrelevant documents
    ranking_docs = ranking.compress_documents(
      query=question,
      documents=docs
    )
    ranking_docs = [doc for doc in ranking_docs if sigmoid(doc.metadata.get("relevance_score", 0.0)) >= DOCUMENT_RELEVANT_THRESHOLD]

    if not ranking_docs:
        return "rewrite"

    else:
        return "generate"



In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

workflow = StateGraph(State)

# Define the nodes
workflow.add_node("agent", agent)
workflow.add_node("retrieve", ToolNode(tools))
workflow.add_node("generate", generate)
workflow.add_node("rewrite", rewrite)

# Build graph
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "retrieve",
        END: END,
    },
)
workflow.add_conditional_edges("retrieve",grade_documents)
workflow.add_edge("rewrite", "agent")
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()


In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# from langchain_community.document_loaders import WebBaseLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter, TokenTextSplitter

# urls = [
#     "https://lilianweng.github.io/posts/2023-06-23-agent/",
#     "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
#     "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
# ]

# # Load
# docs = [WebBaseLoader(url).load() for url in urls]
# docs_list = [item for sublist in docs for item in sublist]

# # Split
# text_splitter = TokenTextSplitter.from_tiktoken_encoder(
#     chunk_size=500, chunk_overlap=0
# )
# doc_splits = text_splitter.split_documents(docs_list)
# vector_store = PGVector(
#         embeddings=embeddings,
#         collection_name=COLLECTION_NAME,
#         connection=CONNECTION,
#         use_jsonb=True,
#     )
# vector_store.add_documents(doc_splits)

In [ ]:
from langchain_core.messages import RemoveMessage, AIMessage, AIMessageChunk

def pretty_print_stream_chunk(msg):
    if isinstance(msg, AIMessageChunk):
        print(msg.content, end="", flush=True)
        if msg.response_metadata.get("finish_reason", "") == "stop":
            print("\n")
    else:
        print(msg.content)
        print("\n")

In [ ]:
from pprint import pprint

# Run
config = {"configurable": {"thread_id": "rag_thread", "user_id": "6746dd846f6d7f728d6cfba7"}}
inputs = "Base on my financial report of November 2024. Can you tell how should I inverst in what Stock? Please retrieve the document I have uploaded"
msg = app.invoke({"messages": inputs}, config=config)

In [ ]:
config = {"configurable": {"thread_id": "rag_thread", "user_id": "6746dd846f6d7f728d6cfba7"}}
inputs = "What is the weather at DANANG City"
msg = app.invoke({"messages": inputs}, config=config)

In [ ]:
for m in msg["messages"]:
    print(m.content)
    print('\n')